# AIMENG D0：Google Colab 训练入口

目标：在 Google Colab 的 GPU 运行时上，对 AIMENG 的 **8,192 神经元、32 维、Top-K 128** 配置做真实语料训练。模型、日志和报告保存到 Google Drive。

**重要：** Colab 免费 GPU 配额和可用时长是动态的，不保证每天固定刷新或固定五小时。先运行预检；如果训练中断，当前训练脚本只在结束时写最终检查点，因此不要把长训练直接当成可中断任务。后续应给训练脚本加周期性检查点/恢复能力。

需要先把 UTF-8 文本语料放到 Google Drive：`MyDrive/aimeng_corpus.txt`。没有真实语料时，只能做 smoke test，不能把合成示例称为语言学习成果。

In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('WARNING: No GPU runtime detected. In Colab choose Runtime > Change runtime type > GPU before training.')

In [ ]:
# Get the exact AIMENG branch this notebook was prepared for.
REPO = '/content/aimeng'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--branch', 'feat/google-colab-d-training', '--single-branch', 'https://github.com/bslsjdk/aimeng.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'feat/google-colab-d-training'], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', 'feat/google-colab-d-training'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
print('Repository ready:', REPO)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CORPUS_PATH = '/content/drive/MyDrive/aimeng_corpus.txt'
RUN_ROOT = '/content/drive/MyDrive/aimeng_runs'
os.makedirs(RUN_ROOT, exist_ok=True)
if not os.path.isfile(CORPUS_PATH):
    raise FileNotFoundError(
        'Real corpus not found: ' + CORPUS_PATH + '\n'
        'Upload a UTF-8 .txt corpus to Google Drive/MyDrive and name it aimeng_corpus.txt. '
        'Do not use the built-in smoke text as evidence of language ability.'
    )
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus_chars = sum(1 for _ in f)
print('Corpus file found:', CORPUS_PATH, '| lines:', corpus_chars)

## D0 预检
先用真实语料跑 20 步，验证 GPU、数据读取、前向/反向传播和检查点保存路径。这个步骤只验证训练链路，不代表模型已掌握语言。预检成功后再运行正式训练。

In [ ]:
SMOKE_OUT = os.path.join(RUN_ROOT, 'D0_preflight')
cmd = [sys.executable, f'{REPO}/scripts/train_diffusion_text.py',
       '--text', CORPUS_PATH, '--neurons', '8192', '--width', '32',
       '--active-k', '128', '--fanout', '8', '--max-steps', '4',
       '--context', '64', '--batch-size', '1', '--steps', '20',
       '--log-every', '1', '--memory-stop-mib', '8192',
       '--output', SMOKE_OUT]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)
print('Preflight report:', os.path.join(SMOKE_OUT, 'report.json'))

## D0 正式训练
默认 500 步作为第一轮基线，先观察速度和验证损失，再决定是否延长。运行时长会因 Colab 分配到的 GPU 和数据量而变化。当前训练脚本仅在这一轮结束时保存检查点；训练中断时不会自动恢复，因此先不要把步数盲目调得很大。

In [ ]:
TRAIN_STEPS = 500
D0_OUT = os.path.join(RUN_ROOT, 'D0_8192n_w32_k128')
os.makedirs(D0_OUT, exist_ok=True)
cmd = [sys.executable, f'{REPO}/scripts/train_diffusion_text.py',
       '--text', CORPUS_PATH, '--neurons', '8192', '--width', '32',
       '--active-k', '128', '--fanout', '8', '--max-steps', '4',
       '--context', '64', '--batch-size', '1', '--steps', str(TRAIN_STEPS),
       '--log-every', '25', '--memory-stop-mib', '8192',
       '--output', D0_OUT]
print('Training config: neurons=8192, width=32, active_k=128, fanout=8, propagation_steps=4, batch_size=1')
subprocess.run(cmd, cwd=REPO, check=True)
print('Checkpoint:', os.path.join(D0_OUT, 'diffusion_checkpoint.pt'))
print('Report:', os.path.join(D0_OUT, 'report.json'))

In [ ]:
import json
report_path = os.path.join(D0_OUT, 'report.json')
with open(report_path, encoding='utf-8') as f:
    report = json.load(f)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('\nInterpretation: lower validation loss than the initial value is a useful first signal, not proof of broad language competence. Check corpus quality and held-out generation before scaling further.')